# 10_dm_fe_and_dyslipidemia

Two supplementary analyses:

1. **Diabetes fixed-effects** verification (conditional logit), mirroring
   notebook 06 for hypertension. Within-person BMI effect is again null.

2. **Dyslipidemia (2024 cross-section)**. The variable exists only in wave
   2024, so no longitudinal verification is possible; we fit a cross-sectional
   risk model as a DiCE demonstration. Note that regular exercise appears as a
   *risk* factor here (reverse causation typical of prevalent cross-sectional
   data) - a concrete illustration of why the longitudinal design is needed.

In [1]:
# 10_dm_fe_and_dyslipidemia.ipynb
# (1) Diabetes within-person fixed-effects verification.

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from statsmodels.discrete.conditional_models import ConditionalLogit

ROOT = os.path.abspath("..")
DATA = os.path.join(ROOT, "data")
RAW  = os.path.join(DATA, "raw")
TAB  = os.path.join(ROOT, "results", "tables")

T = pd.read_parquet(os.path.join(DATA, "khp_transitions.parquet"))
dm = T[T["DM"] == 0].copy()
dm["incident"] = (dm["DM_t1"] == 1).astype(int)
dm = dm.dropna(subset=["BMI", "age", "smoke_cur", "exer_reg"])

cnt   = dm["PIDWON"].value_counts()
multi = cnt[cnt >= 2].index
dmm   = dm[dm["PIDWON"].isin(multi)].copy()
var   = dmm.groupby("PIDWON")["incident"].nunique()
dc    = dmm[dmm["PIDWON"].isin(var[var > 1].index)].copy()

res = ConditionalLogit(dc["incident"].values,
                       dc[["BMI", "smoke_cur", "exer_reg"]].astype(float),
                       groups=dc["PIDWON"].values).fit(disp=0)
OR = np.exp(res.params); ci = np.exp(res.conf_int())
fe = pd.DataFrame({"Variable": ["BMI", "Current smoker", "Regular exercise"],
    "OR":     [OR[v] for v in ["BMI","smoke_cur","exer_reg"]],
    "CI_low": [ci.loc[v,0] for v in ["BMI","smoke_cur","exer_reg"]],
    "CI_high":[ci.loc[v,1] for v in ["BMI","smoke_cur","exer_reg"]],
    "p":      [res.pvalues[v] for v in ["BMI","smoke_cur","exer_reg"]]}).round(3)
fe.to_csv(os.path.join(TAB, "table11_dm_fixed_effects.csv"), index=False)
print(fe.to_string(index=False))
print(f"Informative individuals: {dc['PIDWON'].nunique()}, obs: {len(dc)}")

        Variable    OR  CI_low  CI_high     p
             BMI 1.038   0.937    1.150 0.475
  Current smoker 0.448   0.171    1.173 0.102
Regular exercise 0.882   0.653    1.191 0.414
Informative individuals: 397, obs: 1377


In [2]:
# (2) Dyslipidemia 2024 cross-section (DYS exists only in wave 2024).
import pyreadstat
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

d, _ = pyreadstat.read_sas7bdat(os.path.join(RAW, "y2024", "f_ind.sas7bdat"),
        usecols=["PIDWON","CD1_DYS","HT","WT","S1","S3","P1","P2","BIRTH_Y","SEX"])
d["age"] = 2024 - d["BIRTH_Y"]; d = d[d.age >= 19]
ht = d["HT"].where((d["HT"]>100)&(d["HT"]<250))
wt = d["WT"].where((d["WT"]>20)&(d["WT"]<250))
d["BMI"] = (wt/(ht/100)**2).where(lambda x:(x>10)&(x<60))
d["smoke_cur"] = np.select([d["S3"].isin([1,2]),(d["S3"]==3)|(d["S1"]==3)],[1,0],default=np.nan)
d["exer_reg"]  = np.select([d["P1"]==1,d["P1"]==2],[1,0],default=np.nan)
d["walk_days"] = d["P2"].replace(8,0).mask(d["P2"].isin([-9,-8,-1]))
d["female"]    = (d["SEX"]==2).astype(float)
d["DYS"]       = d["CD1_DYS"].map({1:1, 2:0})
dd = d.dropna(subset=["BMI","age","smoke_cur","exer_reg","walk_days","DYS"])
print(f"Dyslipidemia 2024 sample: {len(dd)}, prevalence {dd['DYS'].mean()*100:.1f}%")

FEATS = ["BMI","age","female","smoke_cur","exer_reg","walk_days"]
Xtr,Xte,ytr,yte = train_test_split(dd[FEATS].astype(float), dd["DYS"].astype(int),
                                   test_size=0.25, random_state=42, stratify=dd["DYS"])
clf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=30,
        class_weight="balanced", random_state=42).fit(Xtr, ytr)
print(f"Cross-sectional risk model AUC = {roc_auc_score(yte, clf.predict_proba(Xte)[:,1]):.3f}")

m = smf.logit("DYS ~ BMI + age + female + smoke_cur + exer_reg", data=dd).fit(disp=0)
OR = np.exp(m.params)
dys = pd.DataFrame({"Variable": ["BMI","Age","Current smoker","Regular exercise"],
    "OR": [OR[v] for v in ["BMI","age","smoke_cur","exer_reg"]],
    "p":  [m.pvalues[v] for v in ["BMI","age","smoke_cur","exer_reg"]]}).round(3)
dys.to_csv(os.path.join(TAB, "table12_dyslipidemia_crosssec.csv"), index=False)
print(dys.to_string(index=False))
print("\nExercise shows as a RISK factor (OR>1): reverse causation in prevalent"
      " cross-sectional data. This is why longitudinal verification is required.")

Dyslipidemia 2024 sample: 12347, prevalence 32.0%


Cross-sectional risk model AUC = 0.767
        Variable    OR     p
             BMI 1.125 0.000
             Age 1.068 0.000
  Current smoker 1.010 0.890
Regular exercise 1.152 0.002

Exercise shows as a RISK factor (OR>1): reverse causation in prevalent cross-sectional data. This is why longitudinal verification is required.
